# VidhanAI — Export fine-tuned LoRA to GGUF (for local Ollama)

**Goal:** merge the QLoRA adapter (`notebooks/lora_model/`, trained by `qlora_finetuning.ipynb`) into the base and quantize it to a **Q4_K_M GGUF (~2 GB)** that can run on your laptop's CPU via Ollama — no GPU needed after this step.

**One-time Kaggle run (~5-10 min on the free T4).**

## Steps before running

1. On your machine, bundle the adapter:
   ```
   python notebooks/roundtrip_lora.py --bundle-adapter
   # -> notebooks/vidhanai_lora_adapter.zip  (wraps lora_model/)
   ```
2. Kaggle -> **Datasets -> New Dataset** -> upload `vidhanai_lora_adapter.zip`. Note the dataset slug.
3. In this notebook: **Add Input** -> attach that dataset (left panel).
4. Run all cells, then **download** the `.gguf` file(s) from the `vidhanai_gguf/` output folder.

Rename the downloaded file to `vidhanai-lora-q4_k_m.gguf`, place it under `D:\models\vidhanai\`, then on the laptop:

```
ollama create vidhanai -f backend/Modelfile.ollama
# model id 'vidhanai' -> set VIDHANAI_USE_OLLAMA=1 and run the app
```

**Honesty note:** this exports the *real* adapter (the 97 MB weights, not the 133-byte stub) — `save_pretrained_gguf` refuses nothing, so sanity-check the adapter path in cell 2 before running.

In [ ]:
# @title Install Unsloth (mirror the training notebook's pins)
!pip install --upgrade --no-deps "transformers>=4.41.0,<4.50.0"
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps peft accelerate bitsandbytes
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('gpu', torch.cuda.get_device_name(0))

In [ ]:
# @title Locate the uploaded adapter under /kaggle/input
from pathlib import Path
import json

adapter_dirs = sorted(p.parent for p in Path('/kaggle/input').rglob('adapter_config.json'))
assert adapter_dirs, 'No adapter found! Attach the vidhanai_lora_adapter dataset (Add Input).'
ADAPTER_DIR = str(adapter_dirs[0])
cfg = json.loads((Path(ADAPTER_DIR) / 'adapter_config.json').read_text())
base = cfg.get('base_model_name_or_path')
print('ADAPTER_DIR =', ADAPTER_DIR)
print('base model  =', base)
print('peft type   =', cfg.get('peft_type'), '| r =', cfg.get('r'))
print('adapter weights (safetensors):')
for p in sorted(Path(ADAPTER_DIR).glob('*.safetensors')):
    print('  ', p.name, '%.1f MB' % (p.stat().st_size / 1e6))

In [ ]:
# @title Load base + adapter, merge, and export Q4_K_M GGUF
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_DIR,     # adapter dir; unsloth loads base + adapter
    max_seq_length=2048,
    dtype=None,                 # auto
    load_in_4bit=True,
)
print('loaded', type(model).__name__)

out_dir = '/kaggle/working/vidhanai_gguf'

# Export. Unsloth's GGUF writer merges the LoRA first. Depending on the
# installed unsloth version, an explicit merge may be required first, so we
# try the direct path and fall back to merge-and-unload.
try:
    model.save_pretrained_gguf(out_dir, tokenizer, quantization_method='q4_k_m')
except AttributeError:
    print('direct GGUF save unavailable -> merging first')
    model = model.merge_and_unload()
    model.save_pretrained_gguf(out_dir, tokenizer, quantization_method='q4_k_m')

print('\nExported files:')
for p in sorted(Path(out_dir).glob('*')):
    print('  ', p.name, '%.2f GB' % (p.stat().st_size / 1e9))

## Next steps (on your machine)

1. Download the `.gguf` file(s) from the `vidhanai_gguf/` output folder (File -> Download, or the Kaggle API).
2. Rename to `vidhanai-lora-q4_k_m.gguf` and move to `D:\models\vidhanai\`.
3. Install Ollama for Windows (free, no card): https://ollama.com/download
4. Register the model:
   ```
   ollama create vidhanai -f backend/Modelfile.ollama
   ollama list
   ```
5. Verify it answers, then run the app with the local fine-tuned model:
   ```
   cd backend
   set VIDHANAI_USE_OLLAMA=1
   python app.py
   ```
   Open http://localhost:5173, click a bill's summary, and confirm the DB stamps `model_version = local_ollama_vidhanai` (see `regenerate_summaries_for_ft.py`).